# kl-shampoo-polar — balancing vs damping intervention (r16)

Causal test of whether **forcing balance** (BaLoRA projection, `LORA_BALANCE_PROJECT=1`) or **more damping** (`precond_delta` 1e-4→1e-3) reduces kl's lr-sensitivity at low rank. Three arms × lr {0.3, 0.1, 0.03}, OLMo-2-1B × opc × r16, 220 steps.

Source: `results/balance_intervention/{base,damp,bal}_lr<lr>.log` (sbatch `slurm_pending/kl_balance_intervention_r16_blackwell.sbatch`). Headline = **final eval_loss vs lr per arm**: lower + flatter across lr = better lr-robustness.

In [ ]:
%load_ext autoreload
%autoreload 2
import json, re, glob, os, math
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
RES = ROOT / 'results' / 'balance_intervention'
ARM_COLOR = {'base': 'tab:gray', 'damp': 'tab:blue', 'bal': 'tab:red'}
ARM_LABEL = {'base': 'base (δ=1e-4)', 'damp': 'damp (δ=1e-3)', 'bal': 'bal (δ=1e-4 + BaLoRA proj)'}

def parse_log(path):
    """Return dict with eval/optim_step/train_norms trajectories from one run log."""
    ev, optim, norms = [], [], []
    for line in open(path, errors='ignore'):
        line = line.strip()
        if not line.startswith('{'):
            continue
        try:
            d = json.loads(line)
        except Exception:
            continue
        e = d.get('event')
        if e == 'eval':
            ev.append((d.get('step'), d.get('eval_loss')))
        elif e == 'optim_step':
            optim.append((d.get('step'), d.get('balance_resid_median')))
        elif e == 'train_norms':
            norms.append((d.get('step'), d.get('param_l2'), d.get('n_non_finite_grads')))
    return {'eval': ev, 'optim': optim, 'norms': norms}

def load_all():
    runs = {}
    for p in sorted(glob.glob(str(RES / '*.log'))):
        m = re.match(r'(base|damp|bal)_lr([0-9.]+)\.log', os.path.basename(p))
        if not m:
            continue
        arm, lr = m.group(1), float(m.group(2))
        runs[(arm, lr)] = parse_log(p)
    return runs

runs = load_all()
print(f'loaded {len(runs)} runs from {RES}')
for k in sorted(runs):
    ev = runs[k]['eval']
    last = ev[-1] if ev else (None, None)
    print(f'  {k}: {len(ev)} evals, last step {last[0]} loss {last[1]}')

In [ ]:
# HEADLINE: final eval_loss vs lr, one line per arm. Lower + flatter = less lr-sensitive.
lrs = sorted({lr for (_, lr) in runs})
fig, ax = plt.subplots(figsize=(7, 5))
for arm in ['base', 'damp', 'bal']:
    xs, ys = [], []
    for lr in lrs:
        r = runs.get((arm, lr))
        if not r or not r['eval']:
            continue
        finite = [(s, l) for s, l in r['eval'] if l is not None and math.isfinite(l)]
        if not finite:
            continue
        xs.append(lr); ys.append(finite[-1][1])
    if xs:
        ax.plot(xs, ys, 'o-', color=ARM_COLOR[arm], label=ARM_LABEL[arm])
ax.set_xscale('log'); ax.set_xlabel('learning rate'); ax.set_ylabel('final eval_loss')
ax.set_title('kl-shampoo-polar r16: final loss vs lr by intervention')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
# balance_resid trajectory per (arm, lr). bal arm should sit ~0 (projection active);
# base/damp climb at high lr if the factors drift off the balanced manifold.
fig, axes = plt.subplots(1, len(lrs), figsize=(5 * len(lrs), 4), sharey=True)
if len(lrs) == 1:
    axes = [axes]
for ax, lr in zip(axes, lrs):
    for arm in ['base', 'damp', 'bal']:
        r = runs.get((arm, lr))
        if not r or not r['optim']:
            continue
        st = [s for s, b in r['optim'] if b is not None]
        bz = [b for s, b in r['optim'] if b is not None]
        ax.plot(st, bz, color=ARM_COLOR[arm], label=ARM_LABEL[arm])
    ax.set_title(f'lr={lr}'); ax.set_xlabel('step'); ax.grid(alpha=0.3)
axes[0].set_ylabel('balance_resid (median over pairs)')
axes[0].legend(fontsize=8)
fig.suptitle('Balance residual trajectory (0 = balanced manifold)')
plt.show()

In [ ]:
# param_l2 trajectory per (arm, lr) — runaway growth is the instability signature.
fig, axes = plt.subplots(1, len(lrs), figsize=(5 * len(lrs), 4), sharey=True)
if len(lrs) == 1:
    axes = [axes]
for ax, lr in zip(axes, lrs):
    for arm in ['base', 'damp', 'bal']:
        r = runs.get((arm, lr))
        if not r or not r['norms']:
            continue
        st = [s for s, p, nf in r['norms'] if p is not None]
        pz = [p for s, p, nf in r['norms'] if p is not None]
        ax.plot(st, pz, color=ARM_COLOR[arm], label=ARM_LABEL[arm])
    ax.set_title(f'lr={lr}'); ax.set_xlabel('step'); ax.grid(alpha=0.3)
axes[0].set_ylabel('param_l2'); axes[0].legend(fontsize=8)
fig.suptitle('param_l2 trajectory (runaway = instability)')
plt.show()